In [1]:
import numpy as np
import SimpleITK as sitk
import skimage as sk
from PIL import Image
from scipy import ndimage
import pandas as pd
import os
import argparse
import time
import matplotlib.pyplot as plt
from pathlib import Path


In [6]:
def plot_registered_tissues(sitk_imgs,map_img,transforms,animal_id,gland,save_path):
    save_img = os.path.join(save_path,f'{animal_id}_{gland}_Registered_Overlay_all_regions.png')
    map_arr  = sitk.GetArrayFromImage(map_img)
    color_list = ['Purples', 'Blues', 'Greens', 'Oranges', 'Reds',
                      'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
                      'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']
    _, axes = plt.subplots()
    axes.imshow(map_arr, cmap="Greys")
    i = 0
    for img, transform in zip(sitk_imgs,transforms):
        resampled = sitk.Resample(
            img, map_img, transform,
            sitk.sitkNearestNeighbor, 0.0, img.GetPixelID()
        )
        resampled_arr  = sitk.GetArrayFromImage(resampled)
        masked_img = np.ma.masked_where(resampled_arr==0,resampled_arr)
        axes.imshow(masked_img,cmap=color_list[i % len(color_list)],alpha=0.7)
        i+=1
    plt.tight_layout()
    plt.savefig(save_img, dpi=600)
    plt.close()

def open_mask(
        img_path: str
    ):
    arr = sk.io.imread(img_path)
    return arr

def open_map(
        img_path: str
    ):
    arr = np.array(Image.open(img_path).convert('L'))
    return arr

def load_sitk_img(img_arr,spacing):
    """ Convert array to 32bit float and then to sitk image for registration """
    bitdepth_map = np.array(img_arr).astype(np.float32)
    sitk_img = sitk.GetImageFromArray(bitdepth_map)
    sitk_img.SetSpacing(spacing)
    return sitk_img

def read_transform(path,img_id,tissue):
    transform_file = os.path.join(path,tissue,'transformation_files','hdf',f'TF_hdf_{img_id}_{tissue}.hdf')
    transform = sitk.ReadTransform(transform_file)
    return transform

def get_animal_ids(
        anno_df
) -> list[str]:
    animal_ids = anno_df['AnimalID'].unique().tolist()
    return animal_ids

In [3]:
path_to_masks = Path('//pn.vai.org/projects/steensma/vari-core-generated-data/Involution_Atlas_collab/KG_Registration_Pipeline/Results_Registration_threepointangle')
save_path = Path("//pn.vai.org/projects/steensma/vari-core-generated-data/Involution_Atlas_collab/KG_Registration_Pipeline/Results_Registration_threepointangle/Registered_Tissue_Overlays_per_animal")
df = pd.read_csv(Path("//pn.vai.org/projects/steensma/vari-core-generated-data/Involution_Atlas_collab/KG_Registration_Pipeline/2026.3.9.AnnotationlevelKey.csv",dtype=str))
df_wt = df[df['Genotype']=='WT']
animal_ids = get_animal_ids(df_wt)

In [ ]:
spacing = (16.1,16.1)
for animal_id in animal_ids:
    animal_df = df_wt[df_wt['AnimalID']==animal_id]
    glands = animal_df['Gland.side'].unique().tolist()
    for gland in glands:
        save_img = os.path.join(save_path,f'{animal_id}_{gland}_Registered_Overlay_all_regions.png')
        if os.path.exists(save_img):
            print(f'overlay already made for {animal_id} {gland} gland, skipping')
            continue
        else:
            animal_gland_df = animal_df[animal_df['Gland.side']==gland]
            gland_map_id = animal_gland_df['MapBase'].unique().tolist()[0]
            gland_map_path = os.path.join(Path("//pn.vai.org/projects/steensma/vari-core-generated-data/Involution_Atlas_collab/KG_Registration_Pipeline/GrayscaleMaps/resized_maps"),f'{gland_map_id}.png')
            if os.path.exists(gland_map_path):
                gland_map_arr = open_map(gland_map_path)
                gland_map_sitk = load_sitk_img(gland_map_arr,spacing)
            else:
                print(f"could not locate {gland_map_path}, skipping {animal_id} {gland} gland")
                continue
            transforms = []
            sitk_imgs = []
            for _, row in animal_gland_df.iterrows():
                image_id = row['Image'] 
                tissue = row['Tissue.ID']
                transform_file_path = os.path.join(f'{path_to_masks}/{tissue}/transformation_files/hdf/TF_hdf_{image_id}_{tissue}.hdf')
                if os.path.exists(transform_file_path):
                    try:
                        transform = read_transform(path_to_masks,image_id,tissue)
                        transforms.append(transform)
                    except ValueError as e:
                            print(f"transform file not found for {image_id} {tissue}, skipping")
                            continue
                else:
                    print('Transform file path not found, skipping')
                    continue
                tissue_img = open_mask(os.path.join(path_to_masks,tissue,'padded_cropped_mask',f'padded_cropped_mask_{image_id}.png'))
                sitk_img = sitk_imgs.append(load_sitk_img(tissue_img,spacing))
                
        if transforms:
            plot_registered_tissues(sitk_imgs,gland_map_sitk,transforms,animal_id,gland,save_path)
        else:
             continue


Transform file path not found, skipping
Transform file path not found, skipping


FileNotFoundError: [Errno 2] No such file or directory: '\\\\pn.vai.org\\projects\\steensma\\vari-core-generated-data\\Involution_Atlas_collab\\KG_Registration_Pipeline\\GrayscaleMaps\\resized_maps\\SD182_Left.png'